In [1]:
import pandas as pd
import sqlite3
import json

In [2]:
customers = pd.DataFrame({
"CustomerID": [
"C001", "C002", "C003", "C004",
"C004", "C005", "C006", "C007",
"C008", "C009", "C010"
],
"CustomerName": [
"Amina", "Brian", "Carol", "David",
"David", "Esther", "Faith", "George",
"Halima", "Ian", "Jane"
],
"Segment": [
"Retail", "Corporate", "Retail", "SME",
"SME", "Corporate", "SME", "Retail",
"Corporate", "Retail", "SME"
],
"RegionCode": [
"NRB", "MSA", "NRB", "KSM",
"KSM", "NKR", "NRB", "MSA",
"KSM", "NKR", "NRB"
]
})
customers


customers.to_csv(
    "customers.csv",
    index=False
)

In [3]:
orders = pd.DataFrame({
"OrderID": [
"O001", "O002", "O003", "O004", "O005",
"O006", "O007", "O008", "O009", "O010",
"O011", "O012", "O013", "O014", "O015"
],
"CustomerID": [
"C001", "C002", "C001", "C003", "C004",
"C005", "C006", "C002", "C007", "C008",
"C009", "C010", "C006", "C003", "C999"
],
"OrderAmount": [
4500, 12500, 3200, 6800, 9100,
15000, 7600, 8300, 2500, 11000,
5400, 6200, 4300, 7200, 9900
],
"ProductCategory": [
"Electronics", "Office", "Home", "Electronics", "Office",
"Electronics", "Home", "Office", "Home", "Electronics",
"Office", "Home", "Electronics", "Office", "Electronics"
]
})
orders

orders.to_csv(
"orders.csv",
index=False
)
orders

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [4]:
regions = [
{"RegionCode": "NRB", "RegionName": "Nairobi"},
{"RegionCode": "MSA", "RegionName": "Mombasa"},
{"RegionCode": "KSM", "RegionName": "Kisumu"},
{"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)


In [5]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

customers.head()
orders.head()
print(customers.shape)   
print(orders.shape)      

(11, 4)
(15, 4)


Q1 A customer row describes a person.

Q2 An order row describes a purchase. 

Q3 one customer has many orders so its different grain cause one customer can have many orders.

In [6]:

customers["CustomerID"].duplicated().sum()
customers[
customers["CustomerID"].duplicated(keep=False)
]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Q4 it will be bad cause it will create duplicate rows and incorrect total sums

In [7]:
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

Q5 C999

Q6 It might tell us info about the data or show us an issue in the dataset 

In [8]:

conn = sqlite3.connect("retail_lab.db")
customers.to_sql(
"customers",
conn,
if_exists="replace",
index=False
)
orders.to_sql(
"orders",
conn,
if_exists="replace",
index=False
)

15

In [9]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [10]:
query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders;
"""
pd.read_sql_query(query, conn)



,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


In [11]:
query = """
SELECT * FROM customers WHERE Segment = 'Retail';"""
pd.read_sql_query(query,conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [12]:
query= """SELECT * FROM customers WHERE Segment = 'Retail' AND CustomerID > 8000;"""
pd.read_sql_query(query,conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [13]:
query= """SELECT * FROM orders WHERE OrderAmount > 8000"""
pd.read_sql_query(query,conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [14]:
query = """SELECT * FROM orders WHERE ProductCategory = 'Electronics'"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [15]:
query = """SELECT * FROM orders
 WHERE OrderAmount > 8000 AND ProductCategory = 'Electronics' AND OrderAmount >8000"""
pd.read_sql_query(query,conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


In [16]:
query = """
SELECT
    ProductCategory,
    COUNT(*)         AS NumOrders,
    SUM(OrderAmount) AS TotalSales,
    AVG(OrderAmount) AS AvgOrderValue
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,NumOrders,TotalSales,AvgOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


Q7 Notice how electronics has high total sales and numbers of orders, meanwhile OFFICE has lower orders but almost equal the average order value

In [17]:
query = """
SELECT o.OrderID, 
o.CustomerID, 
c.CustomerName, 
c.Segment, 
o.OrderAmount
FROM orders o
LEFT JOIN customers c
  ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined



,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [18]:
len(orders)
len(joined)

16

Q8 The order for david was duplicated in the customers table. The query below will return all orders for David, including the duplicated entries.

Q9 The c999 customer in the orders table does not exist in the customers table, which is why it will show NULL for CustomerName and Segment in the result of the LEFT JOIN query.

In [19]:
orders["OrderAmount"].sum()


np.int64(113500)

In [20]:
joined["OrderAmount"].sum()

np.int64(122600)

Q10 The are not eqaul

Q11 Customer C004 david

Q12 Caused the total is wrong and if not fixed will affect analysis by far 

In [21]:
customers_clean = (
customers
.drop_duplicates(subset="CustomerID")
)

In [22]:
customers_clean.to_sql(
"customers_clean",
conn,
if_exists="replace",
index=False
)

10

In [23]:
query = """
SELECT
o.OrderID,
o.CustomerID,
c.CustomerName,
c.Segment,
o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query, conn)
clean_joined


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [24]:
print(len(orders), len(clean_joined))

print(orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum())

15 15
113500 113500


Q13 Yes it has preserved them by removing the duplicating and joining with the orders

In [25]:
query= """ 
SELECT
c.segment,
COUNT(*) AS Orders,
Sum(o.OrderAmount) AS TotalSales,
AVG(o.OrderAmount) AS AvgOrderValue
FROM orders o
LEFT JOIN customers_CLEAN C
ON O.customerID = C.CustomerID
GROUP BY C.segment;
"""

pd.read_sql_query(query, conn)

,Segment,Orders,TotalSales,AvgOrderValue
0,NaN,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


Q14 The retail segment has the most amount of orders but the lowesr average value

In [26]:
regions_df = pd.read_json("regions.json")
regions_df

customers_regions = customers_clean.merge(
regions_df,
on="RegionCode",
how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


In [27]:
regions_df = pd.read_json("regions.json")
regions_df

customers_regions = customers_clean.merge(
regions_df,
on="RegionCode",
how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


Q15 The region name column is filled with every customer

Q16 On a left join when there is no match in the right table it returns NULl

In [28]:
customers_regions.to_sql("customers_regions", conn, 
                         if_exists="replace", 
                         index=False)

query = """
SELECT 
c.RegionName,
COUNT(o.OrderID) AS Orders,
SUM(o.OrderAmount) AS TotalSales
FROM orders o
LEFT JOIN customers_regions c
ON o.CustomerID = c.CustomerID
GROUP BY c.RegionName;
"""
pd.read_sql_query(query, conn)


,RegionName,Orders,TotalSales
0,NaN,1,9900
1,Kisumu,2,20100
2,Mombasa,3,23300
3,Nairobi,7,39800
4,Nakuru,2,20400


Q17 Nairobi

Q18 -no cause nairobi has more customers - and one order has no region

CONCLUSION
The biggest data quality problem was the duplication of the data and the order with customer with no name , cuasing an order to have no regions and segment

It affected the JOIN by giving giving different number of orders due to the duplication 

totals need checking due to the duplicate 
The top segment was corporate with 46800 sales 

One limitation it doesnt give us insight on how the number of orders affected the sales 


In [29]:
#segment with most total sales 
query = """ SELECT
o.OrderID,c.RegionName,SUM(o.OrderAmount) AS Totalsales,c.segment
FROM Orders o
LEFT JOIN
customers_regions c
On o.CustomerID = c.CustomerID
GROUP BY c.RegionName,c.segment
ORDER BY Totalsales DESC
"""
pd.read_sql_query(query, conn)

,OrderID,RegionName,Totalsales,Segment
0,O001,Nairobi,21700,Retail
1,O002,Mombasa,20800,Corporate
2,O007,Nairobi,18100,SME
3,O006,Nakuru,15000,Corporate
4,O010,Kisumu,11000,Corporate
5,O015,NaN,9900,NaN
6,O005,Kisumu,9100,SME
7,O011,Nakuru,5400,Retail
8,O009,Mombasa,2500,Retail


In [30]:
query = """ SELECT c.CustomerName, c.segment, SUM(o.OrderAmount) AS TotalSales
FROM orders o
LEFT JOIN customers_regions c
ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerName, c.segment
ORDER BY TotalSales DESC
"""
pd.read_sql_query(query, conn)

,CustomerName,Segment,TotalSales
0,Brian,Corporate,20800
1,Esther,Corporate,15000
2,Carol,Retail,14000
3,Faith,SME,11900
4,Halima,Corporate,11000
5,NaN,NaN,9900
6,David,SME,9100
7,Amina,Retail,7700
8,Jane,SME,6200
9,Ian,Retail,5400


In [33]:
query= """ SELECT c.RegionName,SUM(o.OrderAmount) AS TotalSales
FROM Orders o
LEFT JOIN customers_regions c
On o.CustomerID = c.CustomerID
GROUP BY c.RegionName
ORDER BY TotalSales DESC"""

pd.read_sql_query(query, conn)

,RegionName,TotalSales
0,Nairobi,39800
1,Mombasa,23300
2,Nakuru,20400
3,Kisumu,20100
4,NaN,9900


In [39]:
query = """ SELECT c.CustomerName,SUM(o.OrderAmount) AS TotalSales
From Orders o
LEFT JOIN customers_regions c
On o.CustomerID = c.CustomerID
GROUP BY c.CustomerName
ORDER BY TotalSales DESC """

pd.read_sql_query(query, conn)

,CustomerName,TotalSales
0,Brian,20800
1,Esther,15000
2,Carol,14000
3,Faith,11900
4,Halima,11000
5,NaN,9900
6,David,9100
7,Amina,7700
8,Jane,6200
9,Ian,5400
